# SegmentAnyTree — Inference Pipeline (Step by Step)

This notebook walks through each step of the inference pipeline individually,
allowing you to inspect intermediate results.

Pipeline steps:
1. File preparation (sanitize names)
2. UTM → local coordinate transform
3. Model inference (PointGroup-PAPER)
4. Result merging (restore UTM coordinates)
5. Visualization

In [ ]:
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path

SAT_ROOT = os.environ.get('SAT_ROOT', '/opt/segmentanytree')
sys.path.insert(0, SAT_ROOT)

INPUT_DIR = Path('/data/input')
OUTPUT_DIR = Path('/data/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 1: File Preparation

In [ ]:
import shutil

work_dir = OUTPUT_DIR / 'input_data'
work_dir.mkdir(exist_ok=True)
for f in INPUT_DIR.iterdir():
    if f.is_file():
        shutil.copy2(f, work_dir)

from sat.pipeline.file_preparation import sanitize_filenames
sanitize_filenames(str(work_dir))
print('Files prepared:', list(work_dir.glob('*')))

## Step 2: UTM → Local Coordinates

In [ ]:
from sat.pipeline.coordinate_transform import utm_to_local_folder

local_dir = OUTPUT_DIR / 'utm2local'
utm_to_local_folder(str(work_dir), str(local_dir))
print('Transformed files:', list(local_dir.glob('*.ply')))

## Step 3: Model Inference

This step runs the PointGroup-PAPER model on the transformed point clouds.

In [ ]:
import shutil
from sat.pipeline.config_update import modify_eval_yaml
from sat.pipeline.cache import clear_inference_cache

# Prepare eval config
eval_yaml = OUTPUT_DIR / 'eval.yaml'
shutil.copy2(Path(SAT_ROOT) / 'conf' / 'eval.yaml', eval_yaml)
modify_eval_yaml(str(eval_yaml), str(local_dir), str(OUTPUT_DIR))
clear_inference_cache(str(eval_yaml))

# Run inference
import subprocess
subprocess.run(['python3', 'eval.py', '--config-name', str(eval_yaml)],
               cwd=SAT_ROOT, check=True)
print('Inference complete!')

## Step 4: Merge Results

In [ ]:
from sat.pipeline.result_rename import rename_instance_results, rename_segmentation_results
from sat.pipeline.result_merge import FolderMerger

rename_instance_results(str(eval_yaml), str(OUTPUT_DIR))
rename_segmentation_results(str(eval_yaml), str(OUTPUT_DIR))

final_dir = OUTPUT_DIR / 'final_results'
FolderMerger(str(local_dir), str(OUTPUT_DIR), str(final_dir), verbose=True).run()
print('Results:', list(final_dir.glob('*')))

## Step 5: Visualize Results

In [ ]:
import laspy
import matplotlib.pyplot as plt
from matplotlib import cm

results = sorted(final_dir.glob('*.las')) + sorted(final_dir.glob('*.laz'))
if results:
    las = laspy.read(str(results[0]))
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Semantic segmentation
    if 'PredSemantic' in las.point_format.dimension_names:
        ax = axes[0]
        ax.scatter(las.x[::10], las.y[::10], c=las.PredSemantic[::10],
                   cmap='Set1', s=0.1, alpha=0.5)
        ax.set_title('Semantic Segmentation')
        ax.set_aspect('equal')

    # Instance segmentation
    if 'PredInstance' in las.point_format.dimension_names:
        ax = axes[1]
        ax.scatter(las.x[::10], las.y[::10], c=las.PredInstance[::10],
                   cmap='tab20', s=0.1, alpha=0.5)
        ax.set_title(f'Instance Segmentation ({len(set(las.PredInstance))} instances)')
        ax.set_aspect('equal')

    plt.suptitle(results[0].name)
    plt.tight_layout()
    plt.show()
else:
    print('No results found')